# Task 5: Production Deployment & Concurrent Load Testing Report

**Model:** APPNP Graph Neural Network
**Framework:** FastAPI (Asynchronous Python)
**Hardware Environment:** Apple Silicon (MPS Profiling)

---

## 1. System Architecture & Implementation
To transition the model from a static notebook into a production-ready environment, we engineered a high-performance inference server with the following components:

* **FastAPI Server:** We deployed the APPNP model using FastAPI, exposing an asynchronous `POST /predict` endpoint designed to accept JSON payloads containing specific Node IDs.
* **100ms Request Batching:** To prevent the GPU from crashing under heavy concurrent traffic, we implemented an `asyncio` queue. The server holds incoming requests in a "waiting room" for exactly 100 milliseconds. Once the timer expires, it bundles all waiting requests into a single tensor batch and processes them simultaneously on the GPU.

---

## 2. Load Testing Methodology & Results
To simulate real-world traffic, we used **Apache Bench (`ab`)** to bombard the `/predict` endpoint. We tested the server against 10, 50, 100, and 500 simultaneous clients to measure throughput and latency.

| Concurrent Clients | Total Requests | Throughput (QPS) | Latency (p50) | Latency (p95) | Max Latency |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **10** | 100 | 29.83 req/sec | 117 ms | 128 ms | 2169 ms* |
| **50** | 500 | 290.04 req/sec | 145 ms | 154 ms | 270 ms |
| **100** | 1,000 | 559.51 req/sec | 164 ms | 177 ms | 177 ms |
| **500** | 5,000 | **1236.84 req/sec** | 379 ms | 609 ms | 817 ms |

> **Note on 10-Client Max Latency:** The initial 2-second spike during the 10-client test was due to the PyTorch MPS "cold start" (loading the model and graph into GPU memory for the very first time). Subsequent requests processed normally.

---

## 3. Hardware Profiling (MPS Memory)
During the load tests, we profiled the hardware using `torch.mps.current_allocated_memory()` (derived from MPS memory stats) to track Megabyte spikes before and after the model's forward pass. 
* By batching the requests, the memory allocation scaled efficiently. The GPU executed the math in a single matrix multiplication rather than hundreds of sequential operations, preventing memory-out-of-bounds errors even at 500 concurrent connections.

---

## 4. Final Report: Bottlenecks & Production Recommendations

* **Max Sustainable QPS:** The server successfully sustained a maximum throughput of **~1,236 Requests Per Second** while remaining completely stable.
* **System Bottlenecks:** Because APPNP relies on a full-graph architecture, the entire CiteSeer adjacency matrix must remain in the GPU's memory. Therefore, the primary bottleneck in a live production environment will be **GPU RAM (VRAM)** during massive batch sizes, rather than CPU processing or network I/O.
* **Recommendations for Production:**
  1. **Dynamic Queue Thresholds:** Currently, the 100ms wait window is static. We recommend adding a "max batch size" trigger (e.g., if the queue hits 256 requests, process immediately without waiting for the 100ms timer to finish) to optimize latency and protect VRAM.
  2. **Model Export:** To further reduce Python overhead and optimize inference speed, the PyTorch weights should be exported to ONNX or CoreML before final deployment.